# Exploración de la EPEN con pandas

Encuesta Permanente de Empleo Nacional, enero a marzo de 2026. Este cuaderno adapta la revisión de estructura, nulos y categorías de `Ejemplo1_Preprocesamiento_solucionado.ipynb` y del laboratorio de clasificación de bosques del curso. Reutiliza el preprocesamiento del experimento.

Ejecuta desde la raíz del repositorio. Si falta el ZIP, usa la descarga indicada en [datos/README.md](datos/README.md) o ejecuta `python experimento.py`. Las celdas muestran resúmenes agregados.

## Carga y filtros

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from experimento import DATA_PATH, FEATURES, LABELS, SEED, load_data

X, y, hogares, auditoria = load_data(DATA_PATH)
pd.Series({
    "Filas del CSV": auditoria["rows_csv"],
    "Columnas del CSV": auditoria["cols_csv"],
    "Edad sin información (código 99), residentes del dominio": auditoria["age_99_in_domain"],
    "Edad superior a 99, residentes del dominio": auditoria["age_above_99_in_domain"],
    "Edad válida desde 14 y residente en el dominio": auditoria["residents_age14_region1"],
    "Etiqueta sin información entre elegibles": auditoria["invalid_target_in_scope"],
    "Personas en la muestra analítica": auditoria["n"],
    "Hogares en la muestra analítica": auditoria["households"],
}, name="Registros")

## Tipos y datos faltantes

El archivo usa códigos numéricos para variables como sexo o educación. Esas variables se tratan como categorías nominales. Se ajustan la codificación y la preparación de la edad usando las filas de entrenamiento de cada pliegue.

In [ ]:
tipo = pd.Series("Categórica", index=FEATURES)
tipo.loc["C208"] = "Numérica"
pd.DataFrame({
    "Tipo para el modelo": tipo,
    "Nulos": X.isna().sum(),
    "Nulos (%)": X.isna().mean().mul(100).round(2),
    "Valores distintos": X.nunique(),
})

In [ ]:
X["C208"].describe().round(2).rename("Edad en años")

## Distribución de clases por partición

El primer pliegue estratificado por hogar sirve como prueba. Una segunda división de cinco pliegues en entrenamiento se usa para elegir hiperparámetros. `OCUP300` conserva las cuatro categorías oficiales y el código las representa internamente de 0 a 3.

In [ ]:
division = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
ajuste, prueba = next(division.split(X, y, hogares))
assert not set(hogares[ajuste]) & set(hogares[prueba])

def conteos(etiquetas):
    return pd.Series(etiquetas).value_counts().reindex(range(4), fill_value=0)

distribucion = pd.DataFrame({
    "Total": conteos(y),
    "Ajuste": conteos(y[ajuste]),
    "Prueba": conteos(y[prueba]),
})
distribucion.index = pd.Index(LABELS, name="Condición laboral")
distribucion["Proporción total (%)"] = distribucion["Total"].div(len(y)).mul(100).round(2)
distribucion

## Resultados en los hogares reservados

Los cuadros resumen la ejecución registrada en `resultados/resultados.json`. El F1 ponderado da más peso a clases con mayor soporte. El F1 macro pesa igual las cuatro clases. Las columnas de sensibilidad indican la proporción de casos reales detectados dentro de cada clase.

In [ ]:
salida = json.loads(Path("resultados/resultados.json").read_text(encoding="utf-8"))
assert salida["support"]["total"] == distribucion["Total"].tolist()
pd.DataFrame.from_dict({
    modelo: {
        "F1 ponderado": fila["f1_weighted"],
        "F1 macro": fila["f1_macro"],
        **{clase: sensibilidad for clase, sensibilidad in zip(LABELS, fila["recall"])},
    }
    for modelo, fila in salida["test"].items()
}, orient="index").round(3)